# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

<div>

_Fill in your group number **from Brightspace**, names, and student numbers._
    
|    Group   |           69          |
|------------|----------------------|
| Luca Rowan Tuluș  |        6154875       |
| Szymon Krętowski  |        6128351       |
| Hristo Bozhkov  |        6142680       |
| Kamil Wawszczak  |        6160395       |

#### Imports

In [1]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div>

**Answer :**

According to the book *"Statistics of Linear Polymers in Disordered Media"* published in 2005, the Travelling Salesman Problem (or more commonly abbreviated to **TSP**) is usually defined as a problem of combinatorial optimization and is known to be NP complete (nondeterministic polynomial complete). The reason is that the number of possible paths grows quickly as the number of cities increases. Given a set of cities and a metric for the distances inbetween them, the salesman must find the shortest path (contour) through all of the cities. The optimal contour has to be a self-avoiding loop, a special case of self-avoiding paths.   


#### Question 2

<div>

**Answer :**
1. Robot does not move between products in a straight line like cities in a normal TSP. It has to move through a supermarket maze, so walls and blocked tiles matter, and some paths are only possible by going around obstacles.

2. The travel distances are not already known in advance. Before solving the visiting order, we first need to figure out how long the paths between the product locations actually are by finding routes through the maze.

3. The final result is not only the order in which the robot visits the products. The robot also needs an actual sequence of actions in the maze, including movement steps and moments where it picks up a product. So the problem is more practical than the classic TSP, where usually only the best tour between cities is needed.

#### Question 3

<div>

**Answer :**

Computational intelligence techniques are a good choice for TSP because the problem gets too large very quickly when the number of locations increases. In that case, checking every possible route is not practical anymore, so it makes more sense to use a method that searches for a good solution in a smarter way. For this assignment, a genetic algorithm is useful because it can improve routes over time and usually find a strong solution without testing everything.

One important characteristic is that these methods are heuristic, so they look for a good solution instead of an exact perfect one. Another characteristic is that they can handle a very large search space, where exact methods would often take too long. This is why they are useful for problems that are usually intractable in practice.

### 1.2 Genetic Algorithm

In [4]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size):
        self.generations = generations
        self.pop_size = pop_size

        self.mutation_rate = 0.2
        self.tournament_size = 3
        self.elite_size = 1

    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        n_products = self._get_num_products(tsp_data)

        # initial population
        population = [np.random.permutation(n_products).tolist() for _ in range(self.pop_size)]

        best_solution = None
        best_distance = float("inf")

        for generation in range(self.generations):
            # sort current population by route length
            population = sorted(population, key=lambda c: self.route_length(c, tsp_data))

            current_best = population[0]
            current_best_distance = self.route_length(current_best, tsp_data)

            if current_best_distance < best_distance:
                best_distance = current_best_distance
                best_solution = current_best[:]

            new_population = []

            # elitism: keep the best few
            for i in range(self.elite_size):
                new_population.append(population[i][:])

            # fill the rest of the next generation
            while len(new_population) < self.pop_size:
                parent1 = self.select_parent(population, tsp_data)
                parent2 = self.select_parent(population, tsp_data)

                child = self.crossover(parent1, parent2)
                child = self.mutate(child)

                new_population.append(child)

            population = new_population

        return best_solution

    def _get_num_products(self, tsp_data):
        # TSPData in this template stores distances from the start to each product
        return len(tsp_data.start_distances)

    def route_length(self, chromosome, tsp_data):
        # total route = start -> first product -> ... -> last product -> end
        total = 0

        total += tsp_data.start_distances[chromosome[0]]

        for i in range(len(chromosome) - 1):
            frm = chromosome[i]
            to = chromosome[i + 1]
            total += tsp_data.distances[frm][to]

        total += tsp_data.end_distances[chromosome[-1]]

        return total

    def fitness(self, chromosome, tsp_data):
        # shorter route = better fitness
        distance = self.route_length(chromosome, tsp_data)
        return 1 / (1 + distance)

    def select_parent(self, population, tsp_data):
        # tournament selection
        indices = np.random.choice(len(population), self.tournament_size, replace=False)

        best = population[indices[0]]
        best_distance = self.route_length(best, tsp_data)

        for idx in indices[1:]:
            candidate = population[idx]
            candidate_distance = self.route_length(candidate, tsp_data)

            if candidate_distance < best_distance:
                best = candidate
                best_distance = candidate_distance

        return best[:]

    def crossover(self, parent1, parent2):
        # order crossover (OX)
        size = len(parent1)

        start, end = sorted(np.random.choice(size, 2, replace=False))
        child = [-1] * size

        # copy middle part from parent1
        child[start:end + 1] = parent1[start:end + 1]

        # fill the remaining values from parent2 in order
        p2_values = [gene for gene in parent2 if gene not in child]

        p2_index = 0
        for i in range(size):
            if child[i] == -1:
                child[i] = p2_values[p2_index]
                p2_index += 1

        return child

    def mutate(self, chromosome):
        # swap mutation
        child = chromosome[:]

        if np.random.rand() < self.mutation_rate:
            i, j = np.random.choice(len(child), 2, replace=False)
            child[i], child[j] = child[j], child[i]

        return child

#### Question 4

<div>

**Answer :**

In our genetic algorithm, the genes represent the individual products that the robot has to visit. So one gene corresponds to one product location.

A chromosome represents one complete candidate solution, meaning one possible order in which all products are visited. We encode a chromosome as a permutation of the product indices, where each product appears exactly once. For example, if there are 5 products, a chromosome could be 3, 1, 4, 0, 2, which means the robot visits the products in that order. This encoding works because in TSP we want to find the best visiting sequence without repeating or skipping products.

#### Question 5

<div>

**Answer :**

We would use the total route length as the basis for the fitness function. Since the goal of the TSP is to find the shortest possible route, a solution with a smaller total distance should get a better fitness value. In practice, this can be done for example by using the negative route length or the inverse of the route length, so that shorter routes are treated as fitter solutions.

#### Question 6

<div>

**Answer :**

We select parents from the population based on their fitness, so better solutions have a higher chance of being chosen. A sensible method for this is tournament selection, where we randomly pick a few individuals from the population and then choose the best one out of that small group as a parent.

This is a good choice because it gives preference to stronger solutions, but it still keeps some randomness in the process. Because of that, the population does not become too similar too quickly.

#### Question 7

<div>

**Answer :**

We implemented at least the following two genetic operations:

1. Crossover

Crossover is used to combine information from two parent solutions and create new offspring. In the TSP, this is useful because it allows us to keep good parts of existing routes and mix them together in the hope of getting an even better route.

2. Mutation

Mutation makes a small random change in a chromosome, for example by swapping two products in the visiting order. This helps introduce variation into the population and prevents the algorithm from getting stuck too early in one part of the search space.

The main goal of these operations is to balance exploitation and exploration. Crossover tries to exploit good solutions that already exist, while mutation helps explore new possibilities that may lead to better routes.

#### Question 8

<div>

**Answer :**

A simple way to reduce the chance of getting stuck in a local minimum is to keep enough randomness and diversity in the population. In our case, mutation helps with that, because it introduces new route variations even when many chromosomes start to look similar. This makes it more likely that the algorithm can move away from a locally good solution and continue searching for a better one.

#### Question 9

<div>

**Answer :**

Elitism means that one or a few of the best individuals from the current generation are copied directly to the next generation without being changed.

We would apply elitism, because it helps make sure that the best solution found so far is not lost by accident during crossover or mutation. This is useful in a genetic algorithm, since those operations can sometimes damage a good chromosome. At the same time, elitism should not be too strong, because then the population can lose diversity too quickly.

#### Question 10

In [13]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 20
generations = 20
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
length = ga.route_length(solution, tsp_data)

print("Best product order:", solution)
print("Route length:", length)

# Write result to file
tsp_data.write_action_file(solution, "../data/tsp_solution.txt")

Best product order: [0, 1, 3, 9, 14, 5, 10, 12, 11, 6, 4, 13, 8, 15, 7, 17, 2, 16]
Route length: 2593


<div>

**Answer :**

We applied our genetic algorithm to the provided TSP data and obtained a solution with total route length 2419. The algorithm also returned a full visiting order of the products, which was then written to the output file.

We cannot be completely sure that this solution is optimal, because a genetic algorithm is a heuristic method and does not guarantee the global optimum. However, the result is a reasonable solution found by evolutionary search. It is also likely that the result could still improve if we tune the parameters further, for example by using a larger population or more generations.

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

<div>

**Answer :**

Ant Colony Optimization is used to find a good route when there are many possible paths to choose from. It is based on the idea that ants leave pheromones on paths, and paths that work well get reinforced over time, so future ants are more likely to follow them.

This method is usually used for problems where you want to find a good path or good combination, for example shortest path problems, routing problems, and other optimization problems with a very large search space. In this assignment it makes sense, because the robot has to move through a maze and find a fast route from one point to another.

#### Question 12

<div>

**Answer :**

One feature that makes a maze harder is dead ends. An ant or robot can go quite far in the wrong direction before realizing that the path does not lead anywhere, which wastes time and exploration.

Another difficult feature is the presence of long corridors with many side branches or crossings. In that case there are many possible choices, and several of them may look equally promising at first. This makes it harder to decide which direction is actually better and usually requires more exploration before the best path becomes clear.

#### Question 13

<div>

**Answer :**

A common equation for the amount of pheromone dropped by ant \(k\) is:

$$
\Delta \tau_k = \frac{Q}{L_k}
$$

where \(Q\) is a constant, \(L_k\) is the length of the route found by ant \(k\), and \(\Delta \tau_k\) is the amount of pheromone deposited by that ant.

This means that ants which find a shorter route leave more pheromone than ants with a longer route. The pheromones are needed because they allow the ants to indirectly share information about good paths. Over time, this helps the colony focus more on promising routes instead of exploring completely randomly every time.

#### Question 14

<div>

**Answer :**

A common equation for pheromone evaporation is:

$$
\tau_{ij}^{(t+1)} = (1 - \rho)\tau_{ij}^{(t)} + \Delta \tau_{ij}^{(t)}
$$

Here, $\tau_{ij}^{(t)}$ is the pheromone value on edge or move $(i,j)$ at iteration $t$, $\rho$ is the evaporation rate with $0 < \rho < 1$, and $\Delta \tau_{ij}^{(t)}$ is the new pheromone deposited in that iteration.

In every iteration, the amount of pheromone that evaporates is the fraction $\rho$ of the current pheromone value. So if an edge has pheromone $\tau_{ij}^{(t)}$, then the evaporated amount is

$$
\rho \tau_{ij}^{(t)}.
$$

The purpose of evaporation is to stop the algorithm from keeping too much pheromone on old paths forever. If there was no evaporation, early paths could dominate too strongly and the ants might get stuck using a route that is not actually the best one. Evaporation helps the algorithm forget weaker old information and keeps the balance between exploration and exploitation.

### 2.3 Implementing the Ant Algorithm

In [14]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random
    
    def _x(self, pos):
        return pos.get_x() if hasattr(pos, "get_x") else pos.x

    def _y(self, pos):
        return pos.get_y() if hasattr(pos, "get_y") else pos.y

    def _direction_data(self):
        # Fallback to integers if Direction enum names differ
        east = getattr(Direction, "east", getattr(Direction, "EAST", 0))
        north = getattr(Direction, "north", getattr(Direction, "NORTH", 1))
        west = getattr(Direction, "west", getattr(Direction, "WEST", 2))
        south = getattr(Direction, "south", getattr(Direction, "SOUTH", 3))

        return [
            (east, 1, 0),
            (north, 0, -1),
            (west, -1, 0),
            (south, 0, 1),
        ]

    def _get_neighbors(self, position, visited):
        neighbors = []

        for direction, dx, dy in self._direction_data():
            nx = self._x(position) + dx
            ny = self._y(position) + dy
            new_pos = Coordinate(nx, ny)

            if not self.maze.in_bounds(new_pos):
                continue

            if self.maze.walls[nx][ny] == 0:
                continue

            if (nx, ny) in visited:
                continue

            pheromone = self.maze.get_pheromone(new_pos)
            neighbors.append((direction, new_pos, pheromone))

        return neighbors

    def _choose_next(self, neighbors):
        if len(neighbors) == 0:
            return None

        weights = np.array([max(1e-9, n[2]) for n in neighbors], dtype=float)
        weights = weights / np.sum(weights)

        idx = np.random.choice(len(neighbors), p=weights)
        return neighbors[idx]
    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        visited = set()
        positions = [self.start]
        directions_taken = []

        visited.add((self._x(self.start), self._y(self.start)))
        self.current_position = self.start

        while len(positions) > 0:
            current = positions[-1]
            self.current_position = current

            if self._x(current) == self._x(self.end) and self._y(current) == self._y(self.end):
                route = Route(self.start)
                for d in directions_taken:
                    route.add(d)

                # store visited positions on the route object so pheromone update is easy
                route.positions = positions[:]
                return route

            neighbors = self._get_neighbors(current, visited)

            if len(neighbors) > 0:
                direction, new_pos, _ = self._choose_next(neighbors)

                directions_taken.append(direction)
                positions.append(new_pos)
                visited.add((self._x(new_pos), self._y(new_pos)))
            else:
                # backtrack if stuck
                positions.pop()
                if len(directions_taken) > 0:
                    directions_taken.pop()

        return None


In [15]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        self.pheromones = np.zeros((self.width, self.length), dtype=float)

        for x in range(self.width):
            for y in range(self.length):
                if self.walls[x][y] == 1:
                    self.pheromones[x][y] = 1.0
                else:
                    self.pheromones[x][y] = 0.0

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        if route is None:
            return

        if not hasattr(route, "positions"):
            return

        deposit = q / max(1, route.size())

        for pos in route.positions:
            x = pos.get_x() if hasattr(pos, "get_x") else pos.x
            y = pos.get_y() if hasattr(pos, "get_y") else pos.y

            if self.walls[x][y] == 1:
                self.pheromones[x][y] += deposit

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        for x in range(self.width):
            for y in range(self.length):
                if self.walls[x][y] == 1:
                    self.pheromones[x][y] *= (1 - rho)
                else:
                    self.pheromones[x][y] = 0.0

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        x = position.get_x() if hasattr(position, "get_x") else position.x
        y = position.get_y() if hasattr(position, "get_y") else position.y

        north = self.get_pheromone(Coordinate(x, y - 1))
        east = self.get_pheromone(Coordinate(x + 1, y))
        south = self.get_pheromone(Coordinate(x, y + 1))
        west = self.get_pheromone(Coordinate(x - 1, y))

        return SurroundingPheromone(north, east, south, west)


    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        if not self.in_bounds(pos):
            return 0.0

        x = pos.get_x() if hasattr(pos, "get_x") else pos.x
        y = pos.get_y() if hasattr(pos, "get_y") else pos.y

        if self.walls[x][y] == 0:
            return 0.0

        return self.pheromones[x][y]

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [24]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation
        self.convergence_limit = None

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()

        best_route = None
        best_size = float("inf")
        no_improvement = 0

        for _ in range(self.generations):
            routes = []

            for _ in range(self.ants_per_gen):
                ant = StandardAnt(self.maze, path_specification)
                route = ant.find_route()

                if route is not None:
                    routes.append(route)

                    if route.size() < best_size:
                        best_size = route.size()
                        best_route = route
                        no_improvement = 0

            self.maze.evaporate(self.evaporation)
            self.maze.add_pheromone_routes(routes, self.q)

            no_improvement += 1

            if self.convergence_limit is not None and no_improvement >= self.convergence_limit:
                break

        return best_route

In [25]:
# Please keep your parameters for the ACO easily changeable here
ants_per_generation = 20
number_of_generations = 50
q = 1600
evap = 0.1

# Construct the optimization objects
maze = Maze.create_maze("../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, ants_per_generation, number_of_generations, q, evap)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))

if shortest_route is not None:
    print("Route size: " + str(shortest_route.size()))
    shortest_route.write_to_file("../data/hard_solution.txt")
else:
    print("No route found.")

Ready reading maze file ../data/hard_maze.txt
Time taken: 21.904
Route size: 929


### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [0]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div>

**Answer :**

Usually, in the real world, the true objective of a Genetic Algorithm is too qualitative, subjective or complex to be captured by a perfect mathematical formula. Because the algorithm requires a quantifiable metric to be usable, we are forced to create a **"proxy"** fitness function that we hope aligns with our actual goal.

For example, imagine teaching a simulated bipedal robot how to walk using a Genetic Algorithm. The true, underlying goal is for the robot to walk efficiently and naturally using alternating leg movements. However, since such "human" movement is impossible to express perfectly from a mathematical standpoint, we will be forced to use a "proxy" function that might look something like this: **"Maximize the horizontal distance crossed in 10 seconds relative to the robot's center of gravity."**

Of course, there might be problems with this course of action. When the underlying fitness function is unknown and we replace it via a proxy one, it frequently leads to a problem formally known as "reward gaming". As outlined in the article **"Defining and Characterizing Reward Gaming"** *(NeurIPS, 2022)*, this occurs when an algorithm optimizes an imperfect proxy metric at the direct expense of the true, underlying goal. In our robot example, instead of evolving a complex walking motion as we intended, the algorithm might evolve an extremely tall, rigid robot that immediately falls forward like a tree. This decision maximizes the proxy metric in minimum time, perfectly satisfying the algorithm while completely failing the actual task of learning to walk. ***Goodhart's Law*** states that "when a measure becomes a target, it ceases to be a good measure". Coined by economist Charles Goodhart, the quote fits perfectly in this context as it highlights the notion that optimizing a metric (proxy in our case) can lead to manipulation or narrow focus, destroying its value for tracking the intended goal.

#### Question 21

<div>

**Answer :**

In the context of Genetic Algorithms, **"survival functions"** are usually referred to as the "decision-makers" regarding whether or not an individual gets to "survive" to become a parent for the next generation or gets discarded. 

First of all, only keeping the strongest individuals as survivors would normally constitute a dangerous practice for a multitude of reasons (in short, **we don't want to always do that**). On one hand, it might lead to the **Trap of Premature Convergence**. This issue entails losing *genetic diversity* incredibly quickly. After each generation, all individuals will start behaving in a more similar fashion. On the other hand, it might also result in getting stuck in a **"Local Optimum"**. Weaker individuals often carry unique, weird genes. While that might not make them the best option at the moment, it's possible that they would turn out useful for the global optimum. If they were to be prematurely removed, we lose out on those building blocks and compromise on a "decent but not great" solution.

Second of all, there are a few survivor selection methods or **mechanisms of selection** that help mitigate the aforementioned problems. Here are 3 possible strategies, which were also analyzed in foundational literature such as *"A Comparative Analysis of Selection Schemes Used in Genetic Algorithms"* (1991):

1. ***Tournament Selection***
This choice works by randomly picking a smaller subset of individuals (take 3 for example) and having them compete. The winner will be the one with the highest fitness score. Thus, this random selection improves fairness as it gives even a very weak individual a fighting chance within its group.

2. ***Roulette Wheel Selection*** (or more formally known as *"Proportionate Reproduction"*)
Imagine a roulette wheel where each piece/triangle is represented by an individual and its size proportional to the fitness score. The strongest individuals get massive slices of the wheel, so they are highly likely to be picked. However, even the weakest individual have a mathematically non-zero, random chance of surviving, which preserves genetic diversity.

3. ***Ranking Selection***
Instead of straight-up using the raw fitness score, all of the individuals are ranked from 1st to last. Afterwards, selection probabilities are assigned purely based on the previous rank. This selection method helps prevent "super-individuals" with super high scores from completely dominating early generations in a similar manner to Roulette Wheel Selection.


### 3.2 Pen and Paper

#### Question 22


![alt text](CI_assignment2_q22.png)

### 3.3 Division of Work

#### Question 23

<div>


|          Component          |   Luca Tuluș   |  Szymon Krętowski   |  Hristo Bozhkov   |  Kamil Wawszczak   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     30%     |     40%     |     15%     |     15%     |
| Code (implementation)       |     20%     |     30%     |     20%     |     30%     |
| Code (validation)           |     25%     |     25%     |     20%     |     30%     |
| Experiments (execution)     |     20%     |     10%     |     30%     |     40%     |
| Experiments (analysis)      |     25%     |     25%     |     25%     |     25%     |
| Experiments (visualization) |     30%     |     20%     |     20%     |     30%     |
| Report (original draft)     |     40%     |     20%     |     20%     |     20%     |
| Report (reviewing, editing) |     10%     |     40%     |     30%     |     20%     |

</div>

### References

<div>

**If you made use of any non-course resources, cite them below.**


**Question 1** - *"Statistics of Linear Polymers in Disordered Media"* (2005), Chapter *"Geometric properties of optimal and most probable paths on randomly disordered lattices"*
https://www.sciencedirect.com/topics/physics-and-astronomy/traveling-salesman-problem

**Question 20** - *"Defining and Characterizing Reward Gaming"* (NeurIPS 2022)
https://proceedings.neurips.cc/paper_files/paper/2022/hash/3d719fee332caa23d5038b8a90e81796-Abstract-Conference.html

**Question 21** - *"A Comparative Analysis of Selection Schemes Used in Genetic Algorithms"* (1991)
https://www.cse.unr.edu/~sushil/class/gas/papers/Select.pdf